# 기준 모델 (물리 기반 + 선형 회귀)

기획서 15:00~16:30 단계. 이후 MLP/GBDT와 비교할 **기준 성능(R², MAE)**을 만든다.

- 데이터: `data_utils.load_clean()` (연속성 깨진 코일 264개 제외, 37,665행)
- 분할: **코일 단위** 80/20 (같은 코일의 5개 스탠드가 학습과 평가에 섞이지 않게)
- 모델
  - 0. 스탠드별 평균 (가장 단순한 기준)
  - 1. 물리 기반: Sims 근사 접촉길이 모델 (속도 정보가 없어 속도항 생략)
  - 2. 물리 기반 + 강종 보정
  - 3. 선형 회귀 (파생 변수 + 스탠드/강종 더미)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GroupShuffleSplit

from data_utils import FORCE, STANDS, add_features, load_clean

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False
sns.set_theme(style="whitegrid", font="Malgun Gothic")

clean, info = load_clean()
print(info)
d = add_features(clean)
print(d.shape)

## 1. 코일 단위 분할

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, te_idx = next(gss.split(d, groups=d["Coil ID"]))
train, test = d.iloc[tr_idx].copy(), d.iloc[te_idx].copy()

assert set(train["Coil ID"]).isdisjoint(test["Coil ID"])  # 코일이 겹치지 않는지 확인
print(f"학습 {len(train)}행 / {train['Coil ID'].nunique()}코일, 평가 {len(test)}행 / {test['Coil ID'].nunique()}코일")
print("평가셋 강종별 코일 수:", test.groupby("Grade")["Coil ID"].nunique().to_dict())

## 2. 평가 함수

In [ ]:
def metrics(y, p):
    y, p = np.asarray(y), np.asarray(p)
    return {
        "R2": r2_score(y, p),
        "MAE (MN)": mean_absolute_error(y, p),
        "MAPE (%)": np.mean(np.abs(y - p) / y) * 100,
    }


results = {}   # 모델 이름 -> 평가셋 예측 (MN)


def evaluate(name, pred_test):
    results[name] = np.asarray(pred_test)
    return metrics(test["force_MN"], pred_test)

## 3. 모델 0: 스탠드별 평균

In [ ]:
stand_mean = train.groupby("Stand ID")["force_MN"].mean()
evaluate("0. 스탠드별 평균", test["Stand ID"].map(stand_mean))

## 4. 모델 1: 물리 기반 (Sims 근사)

롤 분리력 = 평균 변형저항 x 접촉 투영면적 x 형상계수

`F = k x w x L_d x Q_p`

- `L_d = sqrt(R x Δh)`: 접촉길이 근사 (R: 롤 반경, Δh: 압하량)
- `Q_p = 0.785 + 0.25 x L_d / h_m`: Sims 형상계수 근사 (h_m: 입출구 평균 두께)
- `k = C x exp(Q/T) x ε^n`: 변형저항. 온도(T)가 높을수록 작고 변형률(ε)이 클수록 커지는 형태

속도(변형률 속도) 항은 데이터에 없어 뺐다. 양변에 로그를 취하면
`ln(F / (w L_d Q_p)) = ln C + Q x (1/T) + n x ln ε` 가 되어 **파라미터 3개를 최소제곱으로 학습**할 수 있다.

In [ ]:
def physics_terms(x):
    L = x["contact_len"]
    hm = (x["Entry gauge (mm)"] + x["Exit gauge (mm)"]) / 2
    Qp = 0.785 + 0.25 * L / hm
    return x["Width (mm)"] * L * Qp          # mm^2 (접촉 투영면적 x 형상계수)


def phys_design(x, grade_cols=None):
    X = pd.DataFrame({"invT": 1.0 / x["Temperature (K)"], "ln_eps": np.log(x["strain"])}, index=x.index)
    if grade_cols is not None:
        G = pd.get_dummies(x["Grade"]).reindex(columns=grade_cols, fill_value=0).astype(float)
        X = pd.concat([X, G], axis=1)
    X.columns = X.columns.astype(str)
    return X


def fit_physics(use_grade):
    y = np.log(train[FORCE] / physics_terms(train))        # ln(평균 변형저항, Pa)
    cols = sorted(train["Grade"].unique()) if use_grade else None
    X = phys_design(train, cols)
    lr = LinearRegression(fit_intercept=not use_grade).fit(X, y)
    smear = np.mean(np.exp(y - lr.predict(X)))             # 로그 변환 편향 보정

    def predict(x):
        k = np.exp(lr.predict(phys_design(x, cols))) * smear
        return k * physics_terms(x) / 1e6

    return lr, cols, smear, predict


lr1, _, smear1, pred1 = fit_physics(use_grade=False)
print(f"ln C = {lr1.intercept_:.3f}, Q = {lr1.coef_[0]:.0f} K, n = {lr1.coef_[1]:.3f}, smearing = {smear1:.3f}")
evaluate("1. 물리 (Sims)", pred1(test))

## 5. 모델 2: 물리 + 강종 보정
강종마다 변형저항 계수 `C`가 다르다고 보고 강종별 절편을 둔다.

In [ ]:
lr2, gcols2, smear2, pred2 = fit_physics(use_grade=True)
print(f"Q = {lr2.coef_[0]:.0f} K, n = {lr2.coef_[1]:.3f}")
print("강종별 ln C:", dict(zip(gcols2, np.round(lr2.coef_[2:], 3))))
evaluate("2. 물리 + 강종", pred2(test))

## 6. 모델 3: 선형 회귀

In [ ]:
NUM = ["Entry gauge (mm)", "Exit gauge (mm)", "draft", "reduction", "strain", "contact_len",
       "Radius (mm)", "Width (mm)", "Temperature (K)"]
all_grades = sorted(d["Grade"].unique())


def lin_design(x):
    X = pd.concat([
        x[NUM],
        pd.get_dummies(x["Stand ID"]).reindex(columns=STANDS, fill_value=0).astype(float),
        pd.get_dummies(x["Grade"]).reindex(columns=all_grades, fill_value=0).astype(float),
    ], axis=1)
    X.columns = X.columns.astype(str)
    return X


lin = LinearRegression().fit(lin_design(train), train["force_MN"])
evaluate("3. 선형 회귀", lin.predict(lin_design(test)))

## 7. 결과 비교

In [ ]:
table = pd.DataFrame({k: metrics(test["force_MN"], v) for k, v in results.items()}).T
table.round(3)

In [ ]:
# 스탠드별, 강종별 MAE (MN)
def by_group(col):
    out = {}
    for name, p in results.items():
        t = test.assign(pred=p)
        out[name] = t.groupby(col)[["force_MN", "pred"]].apply(
            lambda g: mean_absolute_error(g["force_MN"], g["pred"]))
    return pd.DataFrame(out).round(2)


print("스탠드별 MAE (MN)")
print(by_group("Stand ID").to_string())
print("강종별 MAE (MN)")
print(by_group("Grade").to_string())

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.8), sharex=True, sharey=True)
for a, name in zip(ax, ["1. 물리 (Sims)", "2. 물리 + 강종", "3. 선형 회귀"]):
    t = test.assign(pred=results[name])
    for s in STANDS:
        m = t["Stand ID"] == s
        a.scatter(t.loc[m, "force_MN"], t.loc[m, "pred"], s=4, alpha=0.4, label=s)
    lim = [4, 40]
    a.plot(lim, lim, "k--", lw=1)
    a.set(title=f"{name}  (R2={r2_score(t['force_MN'], t['pred']):.3f})", xlabel="실제 하중 (MN)")
ax[0].set_ylabel("예측 하중 (MN)")
ax[0].legend(markerscale=3)
plt.tight_layout()
plt.show()

## 8. 잔차 확인: 어떤 조건에서 틀리는가

In [ ]:
t = test.assign(res=test["force_MN"] - results["2. 물리 + 강종"])
smp = t.sample(5000, random_state=0)
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
sns.boxplot(data=t, x="Stand ID", y="res", order=STANDS, ax=ax[0], color="#8fb7d9")
ax[0].axhline(0, color="red", lw=1)
ax[0].set(title="스탠드별 잔차 (실제 - 예측, MN)", ylabel="MN")
sns.scatterplot(data=smp, x="reduction", y="res", hue="Stand ID", hue_order=STANDS, s=6, alpha=0.5, ax=ax[1])
ax[1].axhline(0, color="red", lw=1)
ax[1].set(title="압하율에 따른 잔차", ylabel="")
sns.scatterplot(data=smp, x="Temperature (K)", y="res", hue="Stand ID", hue_order=STANDS, s=6, alpha=0.5, ax=ax[2], legend=False)
ax[2].axhline(0, color="red", lw=1)
ax[2].set(title="온도에 따른 잔차", ylabel="")
plt.tight_layout()
plt.show()